Week One Start 10/26/2026

In [32]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import json

from pathlib import Path
from sentence_transformers import SentenceTransformer
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
from sklearn.model_selection import train_test_split

In [33]:
#Import the dataset

from datasets import load_dataset
ds = load_dataset("lmarena-ai/arena-human-preference-140k")

Cleaning the Data

In [34]:
# Since the convos are neded we need to pull out the info
# Build the df

def text_of(content):
  #If it is empty then return empty string
  if content is None:
    return ""

  #if it is just stored as a singular string return the string
  if isinstance(content, str):
    return content

  parts = []

  # if it is saved in a dictionary, then we need to check if what we are dealing with is even
  # textual data, and if it is get the text then append it together and return that
  for c in content:
    if c.get("type") == "text" and c.get("text"):
      parts.append(c["text"])

  return "\n".join(parts)


# checks the messages ordered from newest to oldest, so from end to beginning
# then checks if its from the role we specfify and if it is returns it
def last_msg(conv, role):
  if conv is None:
    conv = []
  for m in reversed(conv):
    if m.get("role") == role:
      return text_of(m.get("content"))
  return ""

#for each conversation we need to get the text that is the prompt and two responses
# the participant can pick from
def extract(batch):
  prompts, resp_a, resp_b = [], [], []

  conv_a = batch["conversation_a"]
  conv_b = batch["conversation_b"]

  for i in range(len(conv_a)):
    prompts.append(last_msg(conv_a[i], "user"))
    resp_a.append(last_msg(conv_a[i], "assistant"))
    resp_b.append(last_msg(conv_b[i], "assistant"))

  return {"prompt": prompts, "resp_a": resp_a, "resp_b": resp_b}

#the columns i want to keep
cols = ["id", "model_a", "model_b", "winner", "evaluation_session_id",
        "evaluation_order", "conversation_a", "conversation_b",
        "category_tag", "language", "is_code", "timestamp"]

#keeps what we want and drops the columns that are very bulky
slim = ds["train"].select_columns(cols).map(
    extract, batched=True, remove_columns=["conversation_a", "conversation_b"])

df = slim.to_pandas()

print(df.shape)

df[["prompt", "resp_a", "resp_b", "winner", "language"]].head()

#Summary of incorrect or missing data points
print("Duplicate IDs:", df.id.duplicated().sum())
print("Bad winner:", (~df.winner.isin(["model_a","model_b","tie","both_bad"])).sum())
print("Missing session:", df.evaluation_session_id.isna().sum())
print("Empty prompt:", (df.prompt.str.strip() == "").sum())
print("Empty resp_a:", (df.resp_a.str.strip() == "").sum())
print("Empty resp_b:", (df.resp_b.str.strip() == "").sum())
print("Missing language:", df.language.isna().sum())

(135634, 13)
Duplicate IDs: 0
Bad winner: 0
Missing session: 0
Empty prompt: 0
Empty resp_a: 74
Empty resp_b: 65
Missing language: 0


In [35]:
#Prints out the summary table of our information

print("RAW ROWS:", len(df))
print("\nRAW LABEL SPLIT:\n", df.winner.value_counts())
print("\nRAW LANGUAGE MIX (top 10):\n", df.language.value_counts().head(10))
print("\nDROP COUNTS:")
print(" duplicate ids:", df.id.duplicated().sum())
print(" bad winner:", (~df.winner.isin(["model_a","model_b","tie","both_bad"])).sum())
print(" missing session:", df.evaluation_session_id.isna().sum())
print(" empty prompt:", (df.prompt.str.strip() == "").sum())
print(" empty resp (A or B):", ((df.resp_a.str.strip() == "") | (df.resp_b.str.strip() == "")).sum())

RAW ROWS: 135634

RAW LABEL SPLIT:
 winner
model_b     49785
model_a     48563
tie         21532
both_bad    15754
Name: count, dtype: int64

RAW LANGUAGE MIX (top 10):
 language
en     71175
pl     13813
und    11835
ru      9263
zh      6501
de      4311
ko      2486
ja      2444
fr      2128
pt      1684
Name: count, dtype: int64

DROP COUNTS:
 duplicate ids: 0
 bad winner: 0
 missing session: 0
 empty prompt: 0
 empty resp (A or B): 136


In [36]:
#Crosstab cell, checks in who wins in the case of an empty model
#needed in order to see if we can drop the empty rows

#We want to see if the empty ones actually do something in our data
# so lets see if anyone actually picked an empty response over a response

a_blank = df.resp_a.str.strip() == ""
b_blank = df.resp_b.str.strip() == ""

a_empty = df[a_blank & ~b_blank]
b_empty = df[~a_blank & b_blank]
both_empty = df[a_blank & b_blank]

print("A empty, B not:\n", a_empty.winner.value_counts())
print("B empty, A not:\n", b_empty.winner.value_counts())
print("Both empty:", len(both_empty))

A empty, B not:
 winner
model_b     49
model_a     11
both_bad     8
tie          3
Name: count, dtype: int64
B empty, A not:
 winner
model_a     45
both_bad    16
tie          1
Name: count, dtype: int64
Both empty: 3


In [37]:
#Drop the columns that aren't in english, bc we can't use non-english rows

df = df[(df.resp_a.str.strip() != "") & (df.resp_b.str.strip() != "")].copy()
df_en = df[df.language == "en"].reset_index(drop=True)

print(df.language.value_counts().head(10))
print(len(df), len(df_en))


language
en     71102
pl     13798
und    11822
ru      9253
zh      6494
de      4306
ko      2486
ja      2443
fr      2127
pt      1684
Name: count, dtype: int64
135498 71102


In [38]:
#EDA
#Note the new data set being used is called df_en bc
# that is where we saved our dataset with all rows that are only in english

Path("data").mkdir(exist_ok=True)

# Grab the length of the features
df_en["prompt_len"] = df_en.prompt.str.len()
df_en["len_a"] = df_en.resp_a.str.len()
df_en["len_b"] = df_en.resp_b.str.len()

# Get the value of which model won each time
print("LABEL SPLIT, English:")
print(df_en.winner.value_counts(), "\n")
#Get the percents
print(df_en.winner.value_counts(normalize=True).round(4), "\n")

# prompt lengths
print("PROMPT LENGTH (chars), English:")
#
print(df_en.prompt_len.describe(percentiles=[.05, .25, .5, .75, .95, .99]), "\n")

# longer-response baseline (English, decisive votes, unequal lengths)
decider = df_en[df_en.winner.isin(["model_a", "model_b"])]
decider = decider[decider.len_a != decider.len_b]
acc = ((decider.len_a > decider.len_b) == (decider.winner == "model_a")).mean()
print(f"LONGER-RESPONSE BASELINE: {acc:.4f} on n = {len(decider)}")

# save for dataset
out = df_en.copy()
out["category_tag"] = out.category_tag.apply(json.dumps)
out.to_parquet("data/clean_en.parquet", index=False)
print("saved", out.shape)

print((df_en.prompt_len > 1000).mean())

LABEL SPLIT, English:
winner
model_b     26052
model_a     25553
tie         11373
both_bad     8124
Name: count, dtype: int64 

winner
model_b     0.3664
model_a     0.3594
tie         0.1600
both_bad    0.1143
Name: proportion, dtype: float64 

PROMPT LENGTH (chars), English:
count     71102.000000
mean        433.046145
std        2146.597479
min           1.000000
5%           22.000000
25%          47.000000
50%          89.000000
75%         212.000000
95%        1341.000000
99%        7247.910000
max      119164.000000
Name: prompt_len, dtype: float64 

LONGER-RESPONSE BASELINE: 0.6002 on n = 51552
saved (71102, 16)
0.06433011729627859


In [39]:
#in our parquet (saved into a parquet bc that what the original data was stored in)
# we should see
# (71102, 16)
# ['en']
#if we did this correctly

chk = pd.read_parquet("data/clean_en.parquet")
print(chk.shape)
print(chk.language.unique())
print(chk.winner.value_counts())

(71102, 16)
['en']
winner
model_b     26052
model_a     25553
tie         11373
both_bad     8124
Name: count, dtype: int64


Embedding

In [40]:

#only take in up to 1000 characters, this is done primarily bc all-MiniLM-L6-v2
#can only take in 256 tokens, so 1000 is just about its limit

text = df_en.prompt.str.slice(0, 1000)

# one embedding per distinct prompt
codes, uniques = pd.factorize(text)
print(len(df_en), "rows ->", len(uniques), "distinct prompts")

#embed it
model = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")
emb = model.encode(list(uniques), batch_size=128, show_progress_bar=True,
                   normalize_embeddings=True).astype("float32")

#save the outputs so we can use and look at them later
np.save("data/embeddings.npy", emb)
pd.DataFrame({"id": df_en.id.values, "emb_row": codes}) \
    .to_parquet("data/embedding_index.parquet", index=False)
print(emb.shape)

71102 rows -> 67332 distinct prompts


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Batches:   0%|          | 0/527 [00:00<?, ?it/s]

(67332, 384)


In [41]:
# set up k-means split

Path("results").mkdir(exist_ok=True)

#need to split bu sessions not rows
sessions = df_en.evaluation_session_id.unique()

train, hold_out = train_test_split(sessions, test_size=0.30, random_state=42)
validation, test = train_test_split(hold_out, test_size=0.50, random_state=42)

#Label it so its easy to see when we print it out
df_en["split"] = "test"
df_en.loc[df_en.evaluation_session_id.isin(train), "split"] = "train"
df_en.loc[df_en.evaluation_session_id.isin(validation), "split"] = "validation"

print(df_en.groupby("split").evaluation_session_id.nunique())

split
test           9224
train         43045
validation     9224
Name: evaluation_session_id, dtype: int64


In [42]:
df_en[["id", "evaluation_session_id", "split"]].to_parquet("data/splits.parquet", index=False)
print("saved splits:", df_en.split.value_counts().to_dict())

saved splits: {'train': 49777, 'test': 10681, 'validation': 10644}


In [43]:
#Session Count
decisive = df_en[df_en.winner.isin(["model_a", "model_b"])]

#Group by which side wins, and then see which side it falls on
sess = decisive.groupby("evaluation_session_id").agg(n=("winner", "size"),
                                                split=("split", "first"))

groups = {"All": sess,
          "Test": sess[sess.split == "test"]}

rows = []

#try and get an idea where there is enough test and training data
# k+1 bc we are assuming if we have at least 2 votes, so if k = 1
# then we need one more in order to test.
for name, group in groups.items():
  for k in [0, 1, 2, 3, 4, 5, 10, 20]:
    e = group[group.n >= k + 1]
    rows.append({"Group": name,
                 "k": k,
                 "Sessions with Enough Votes": len(e),
                 "Votes left after K": int((e.n - k).sum())})


table = pd.DataFrame(rows)
print(table)
table.to_csv("results/session_counts.csv", index=False)

   Group   k  Sessions with Enough Votes  Votes left after K
0    All   0                       45778               51605
1    All   1                        3985                5827
2    All   2                        1035                1842
3    All   3                         412                 807
4    All   4                         180                 395
5    All   5                          93                 215
6    All  10                           7                  23
7    All  20                           0                   0
8   Test   0                        6842                7739
9   Test   1                         608                 897
10  Test   2                         164                 289
11  Test   3                          66                 125
12  Test   4                          30                  59
13  Test   5                          15                  29
14  Test  10                           0                   0
15  Test  20            

In [47]:
# K-Mean Sweep
# emb = np.load("data/embeddings.npy")
# codes = pd.read_parquet("data/embedding_index.parquet").emb_row.values

# one embedding per English vote
X = emb[codes]
X_train = X[(df_en.split == "train").values]

results = []
for K in range(4, 13):
    km = KMeans(n_clusters=K, n_init=3, random_state=42).fit(X_train)
    sil = silhouette_score(X_train, km.labels_, sample_size=10000, random_state=42)
    sizes = np.bincount(km.labels_) / len(X_train)
    results.append({"K": K, "inertia": km.inertia_, "silhouette": sil,
                    "smallest_cluster": sizes.min(), "largest_cluster": sizes.max()})
    print(results[-1])

pd.DataFrame(results).to_csv("results/kmeans_sweep.csv", index=False)

{'K': 4, 'inertia': 46188.5859375, 'silhouette': np.float32(0.010537338), 'smallest_cluster': np.float64(0.14673443558269883), 'largest_cluster': np.float64(0.40751752817566345)}
{'K': 5, 'inertia': 45852.921875, 'silhouette': np.float32(0.0067695724), 'smallest_cluster': np.float64(0.0486570102657854), 'largest_cluster': np.float64(0.36410390340920507)}
{'K': 6, 'inertia': 45564.90625, 'silhouette': np.float32(0.008284736), 'smallest_cluster': np.float64(0.04743154468931434), 'largest_cluster': np.float64(0.26126524298370735)}
{'K': 7, 'inertia': 45300.15234375, 'silhouette': np.float32(0.009138131), 'smallest_cluster': np.float64(0.04287120557687285), 'largest_cluster': np.float64(0.2218293589408763)}
{'K': 8, 'inertia': 45070.34765625, 'silhouette': np.float32(0.0100008305), 'smallest_cluster': np.float64(0.04208771119191595), 'largest_cluster': np.float64(0.1733531550716194)}
{'K': 9, 'inertia': 44871.4375, 'silhouette': np.float32(0.01119559), 'smallest_cluster': np.float64(0.0413

In [46]:
df.winner.value_counts().rename_axis("winner").reset_index(name="votes") \
    .to_csv("results/eda_label_split_all.csv", index=False)
df_en.winner.value_counts().rename_axis("winner").reset_index(name="votes") \
    .to_csv("results/eda_label_split_en.csv", index=False)
df.language.value_counts().rename_axis("language").reset_index(name="votes") \
    .to_csv("results/eda_language.csv", index=False)
df_en.prompt_len.describe(percentiles=[.05, .25, .5, .75, .95, .99]) \
    .to_csv("results/eda_prompt_lengths_en.csv")

Week 1 done